In [ ]:
import os
import re
import time
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

# ---------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------
MODEL_NAME = "sarvamai/sarvam-1"
MAX_REASONING_TOKENS = 3000
INTERVAL = 50
QUESTIONS_PER_LANG = 200
OUTPUT_CSV = "gsm8k_sarvam1_2b_interval_results.csv"

LANGUAGES = [
    "english", "hindi", "bengali", "tamil", "telugu", 
    "marathi", "gujarati", "malayalam", "kannada", "punjabi", "odia"
]

LANG_CONFIGS = {
    "english": "en", "hindi": "hi", "bengali": "bn", "tamil": "ta",
    "telugu": "te", "marathi": "mr", "gujarati": "gu", "malayalam": "ml",
    "kannada": "kn", "punjabi": "pa", "odia": "or"
}

LANG_PREFIXES = {
    "english": "EN", "hindi": "HI", "bengali": "BN", "tamil": "TA",
    "telugu": "TE", "marathi": "MR", "gujarati": "GU", "malayalam": "ML",
    "kannada": "KN", "punjabi": "PA", "odia": "OR"
}

# Unicode Script Ranges for Target Language Tracking
SCRIPT_REGEX = {
    "english": r'[a-zA-Z]',
    "hindi": r'[\u0900-\u097F]',
    "bengali": r'[\u0980-\u09FF]',
    "tamil": r'[\u0B80-\u0BFF]',
    "telugu": r'[\u0C00-\u0C7F]',
    "marathi": r'[\u0900-\u097F]',
    "gujarati": r'[\u0A80-\u0AFF]',
    "malayalam": r'[\u0D00-\u0D7F]',
    "kannada": r'[\u0C80-\u0CFF]',
    "punjabi": r'[\u0A00-\u0A7F]',
    "odia": r'[\u0B00-\u0B7F]'
}

# Native "Let's think step by step:"
COT_PROMPTS = {
    "english": "\nLet's think step by step:\n",
    "hindi": "\nआइए कदम दर कदम सोचें:\n",
    "bengali": "\nআসুন ধাপে ধাপে চিন্তা করি:\n",
    "tamil": "\nபடிப்படியாக சிந்திப்போம்:\n",
    "telugu": "\nదశలవారీగా ఆలోచిద్దాం:\n",
    "marathi": "\nचला टप्प्याटप्प्याने विचार करूया:\n",
    "gujarati": "\nચાલો પગલું દ્વારા પગલું વિચારીએ:\n",
    "malayalam": "\nനമുക്ക് പടിപടിയായി ചിന്തിക്കാം:\n",
    "kannada": "\nಹಂತ ಹಂತವಾಗಿ ಯೋಚಿಸೋಣ:\n",
    "punjabi": "\nਆਓ ਕਦਮ-ਦਰ-ਕਦਮ ਸੋਚੀਏ:\n",
    "odia": "\nଆସନ୍ତୁ ପର୍ଯ୍ୟାୟକ୍ରମେ ଚିନ୍ତା କରିବା:\n"
}

# Native "Therefore, the answer is "
ANSWER_TRIGGERS = {
    "english": "\nTherefore, the answer is ",
    "hindi": "\nइसलिए, उत्तर है ",
    "bengali": "\nঅতএব, উত্তর হল ",
    "tamil": "\nஎனவே, பதில் ",
    "telugu": "\nకాబట్టి, సమాధానం ",
    "marathi": "\nम्हणून, उत्तर आहे ",
    "gujarati": "\nતેથી, જવાબ છે ",
    "malayalam": "\nഅതിനാൽ, ഉത്തരം ",
    "kannada": "\nಆದ್ದರಿಂದ, ಉತ್ತರ ",
    "punjabi": "\nਇਸ ਲਈ, ਜਵਾਬ ਹੈ ",
    "odia": "\nତେଣୁ, ଉତ୍ତର ହେଉଛି "
}

# ---------------------------------------------------------
# 2. Helper Functions
# ---------------------------------------------------------
def calculate_script_ratio(text, lang):
    """Calculates character-level target script ratio, excluding spaces, digits & punctuation."""
    pattern = SCRIPT_REGEX.get(lang, r'[a-zA-Z]')
    counted_chars = re.sub(r'[\s\d\W_]', '', text)
    if not counted_chars:
        return 0.0
    target_chars = re.findall(pattern, counted_chars)
    return round((len(target_chars) / len(counted_chars)) * 100.0, 2)

def parse_answer(text):
    """Extracts final integer from answer text."""
    clean_text = text.split('\n')[0].strip()
    numbers = re.findall(r'-?\d+', clean_text.replace(',', ''))
    return int(numbers[-1]) if numbers else None

# ---------------------------------------------------------
# 3. Model Initialization
# ---------------------------------------------------------
print(f"Loading tokenizer and model: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)
model.eval()

# ---------------------------------------------------------
# 4. Initialize CSV Header
# ---------------------------------------------------------
FIELDNAMES = [
    "sample_id", "question_id", "language", "model", "question",
    "max_cutoff_limit", "tokens_used", "full_thinking_text",
    "raw_answer_text", "parsed_answer", "ground_truth", "correct",
    "step1_time_sec", "step2_time_sec", "target_lang_ratio_pct", "cutoff_status"
]

if not os.path.exists(OUTPUT_CSV):
    pd.DataFrame(columns=FIELDNAMES).to_csv(OUTPUT_CSV, index=False)

# Check existing records for safe resumption
existing_df = pd.read_csv(OUTPUT_CSV)
completed_samples = set(existing_df['sample_id'].unique()) if not existing_df.empty else set()
print(f"Resuming evaluation... ({len(completed_samples)} samples already logged)")

# ---------------------------------------------------------
# 5. Main Evaluation Loop
# ---------------------------------------------------------
for lang in LANGUAGES:
    print("\n" + "="*80)
    print(f"LANGUAGE: {lang.upper()} (Config: {LANG_CONFIGS[lang]})")
    print("="*80)
    
    config_code = LANG_CONFIGS[lang]
    lang_prefix = LANG_PREFIXES[lang]
    
    try:
        dataset = load_dataset("sarvamai/gsm8k-indic", config_code, split="test")
    except Exception as e:
        print(f"Error loading dataset for {lang}: {e}")
        continue

    sample_count = min(QUESTIONS_PER_LANG, len(dataset))

    for q_idx in range(sample_count):
        item = dataset[q_idx]
        question_text = item['question']
        
        # Parse ground truth
        raw_answer_str = item['answer']
        final_answer_str = raw_answer_str.split('####')[-1].strip()
        ground_truth = int(final_answer_str.replace(',', ''))
        
        question_id = f"GSM8K_{lang_prefix}_{q_idx + 1}"
        
        # Language-Specific Prompts
        cot_prompt = COT_PROMPTS[lang]
        trigger_text = ANSWER_TRIGGERS[lang]
        
        prompt = f"{question_text}{cot_prompt}"
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
        trigger_ids = tokenizer(trigger_text, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
        
        # -----------------------------------------------------
        # STEP 1: Generate Full Reasoning Trace (3000 Tokens)
        # -----------------------------------------------------
        t0_step1 = time.time()
        with torch.no_grad():
            full_outputs = model.generate(
                input_ids,
                max_new_tokens=MAX_REASONING_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        step1_time = round(time.time() - t0_step1, 3)
        
        generated_reasoning_ids = full_outputs[0][input_ids.shape[1]:]
        total_tokens_used = len(generated_reasoning_ids)
        
        # Determine target budget intervals
        cutoffs = list(range(INTERVAL, MAX_REASONING_TOKENS + INTERVAL, INTERVAL))
        
        # Cache variables for when model finishes naturally
        cached_finished_data = None
        
        question_rows = []
        
        # -----------------------------------------------------
        # STEP 2: Interval Slicing & Probing
        # -----------------------------------------------------
        for cutoff in cutoffs:
            sample_id = f"{question_id}_b{cutoff}"
            
            if sample_id in completed_samples:
                continue
                
            # Check if we reached or exceeded actual trace length
            if cutoff >= total_tokens_used:
                cutoff_status = "Finished_Thinking_Naturally"
                actual_cutoff_tokens = total_tokens_used
            else:
                cutoff_status = "Budget_Exhausted"
                actual_cutoff_tokens = cutoff
                
            # REUSE CACHED RESULT if thinking finished naturally in an earlier cutoff
            if cached_finished_data is not None and cutoff_status == "Finished_Thinking_Naturally":
                row_data = cached_finished_data.copy()
                row_data["sample_id"] = sample_id
                row_data["max_cutoff_limit"] = cutoff
                question_rows.append(row_data)
                continue
                
            # Slice latent reasoning trace
            sliced_ids = generated_reasoning_ids[:actual_cutoff_tokens]
            if len(sliced_ids) > 0 and sliced_ids[-1] == tokenizer.eos_token_id:
                sliced_ids = sliced_ids[:-1]
                
            sliced_thinking_text = tokenizer.decode(sliced_ids, skip_special_tokens=True)
            target_ratio = calculate_script_ratio(sliced_thinking_text, lang)
            
            # Prepare Step 2 Input: [Prompt] + [Sliced Reasoning] + [Answer Trigger]
            sliced_ids_tensor = sliced_ids.unsqueeze(0)
            probe_input_ids = torch.cat([input_ids, sliced_ids_tensor, trigger_ids], dim=1)
            
            t0_step2 = time.time()
            with torch.no_grad():
                answer_outputs = model.generate(
                    probe_input_ids,
                    max_new_tokens=15,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )
            step2_time = round(time.time() - t0_step2, 3)
            
            generated_answer_ids = answer_outputs[0][probe_input_ids.shape[1]:]
            raw_answer_text = tokenizer.decode(generated_answer_ids, skip_special_tokens=True).split('\n')[0].strip()
            
            parsed_pred = parse_answer(raw_answer_text)
            is_correct = (parsed_pred == ground_truth) if parsed_pred is not None else False
            
            row_data = {
                "sample_id": sample_id,
                "question_id": question_id,
                "language": lang,
                "model": MODEL_NAME,
                "question": question_text,
                "max_cutoff_limit": cutoff,
                "tokens_used": actual_cutoff_tokens,
                "full_thinking_text": sliced_thinking_text,
                "raw_answer_text": raw_answer_text,
                "parsed_answer": parsed_pred if parsed_pred is not None else "",
                "ground_truth": ground_truth,
                "correct": is_correct,
                "step1_time_sec": step1_time,
                "step2_time_sec": step2_time,
                "target_lang_ratio_pct": target_ratio,
                "cutoff_status": cutoff_status
            }
            
            question_rows.append(row_data)
            
            # Store cache if this was the first naturally completed cutoff
            if cutoff_status == "Finished_Thinking_Naturally" and cached_finished_data is None:
                cached_finished_data = row_data
                
        # Append batch to CSV file
        if question_rows:
            batch_df = pd.DataFrame(question_rows)
            batch_df.to_csv(OUTPUT_CSV, mode='a', header=False, index=False)
            
        status_icon = "✅" if any(r["correct"] for r in question_rows if r["cutoff_status"] == "Finished_Thinking_Naturally") else "❌"
        print(f"  [{question_id}] Tokens Used: {total_tokens_used:4d} | Step1 Time: {step1_time:.2f}s | Result: {status_icon}")

print(f"\nEvaluation Complete! Results logged to '{OUTPUT_CSV}'.")